# Draws read-off 2 — WV6 excursion test, bloc homogeneity, dispersion trajectory

**Run 2026-07-16. Read-only arithmetic — no model fit or refit, no existing notebook executed.**
All quantities computed from `results/bootstrap_draws.parquet` (replicate-paired draws) and
`results/closing_checks_table.csv` / `results/draws_readoff_table.csv`. Point estimates are the
stored full-sample values; SEs/CIs/percentile p's come from paired draws. New files: this
notebook, `results/draws_readoff2_table.csv`.

## Verdicts

1. **WV6:** the army's 2011 gradient is **not distinguishable from its WV5/WV7 neighbours** — joint χ²(2) = 2.44, **p = 0.295** (WV6−WV5 = −0.244, p = .32; WV6−WV7 = −0.251, p = .15). The comparison case police **is** distinguishable (χ²(2) = 8.04, p = .018, driven by WV6−WV7 = −0.398, p = .012); the negative control courts shows nothing (χ²(2) = 0.01, p = .997).
2. **BLOC:** at WV2 the six gradients **are (marginally) statistically distinguishable** — χ²(5) = 11.43, **p = 0.044** (6/6 exclude zero; spread 0.497, government −0.43 vs parliament −0.93); at WV7 they are **overwhelmingly distinguishable** — χ²(5) = 81.20, **p < 10⁻⁵** (3/6 exclude zero).
3. **DISPERSION:** SD(WV7) − SD(WV2) = **+0.040, p = 0.831** — 2018 dispersion is *not* significantly larger than 1990 — and the **series minimum is WV3 (1996)**, not 2001 (five-institution series: 0.183, 0.075, 0.087, 0.197, 0.090, 0.222); the only significant paired rise is WV7 vs WV4 (+0.135, p = .026).


In [1]:
# GATE: parquet shape + WV2 LEVEL cross-table match to 4 dp
import numpy as np
import pandas as pd
from scipy import stats

R = "results"
WAVES = ["WV2", "WV3", "WV4", "WV5", "WV6", "WV7"]
INST6 = ["military", "police", "courts", "parliament", "government", "civil_service"]
INST5 = [i for i in INST6 if i != "courts"]

dr = pd.read_parquet(f"{R}/bootstrap_draws.parquet")
cc = pd.read_csv(f"{R}/closing_checks_table.csv")
ro = pd.read_csv(f"{R}/draws_readoff_table.csv")
eq = cc[cc["check"] == "1_equivalence"].set_index(["institution", "wave"])
ro_eq = ro[ro["quantity"] == "equivalence_both_margins"].set_index(["institution", "wave"])
piv = dr.pivot_table(index="replicate", columns=["institution", "wave"], values="value")
print(f"parquet shape: {dr.shape} (expected (34958, 5))")
assert dr.shape == (34958, 5), "parquet shape mismatch — STOP"

ROWS = []
def add(quantity, inst, wave, estimate, boot_se=np.nan, ci90=(np.nan, np.nan),
        ci95=(np.nan, np.nan), p=np.nan, n=np.nan, notes=""):
    ROWS.append({"quantity": quantity, "institution": inst, "wave": wave,
                 "estimate": estimate, "boot_se": boot_se,
                 "ci90_lo": ci90[0], "ci90_hi": ci90[1],
                 "ci95_lo": ci95[0], "ci95_hi": ci95[1],
                 "p": p, "n_draws": n, "notes": notes})

print("\nWV2 LEVELs across tables (must match to 4 dp):")
for inst in INST6:
    a = float(eq.loc[(inst, "WV2"), "estimate"])
    b = float(ro_eq.loc[(inst, "WV2"), "estimate"])
    ok = abs(a - b) < 5e-5
    med = float(piv[(inst, "WV2")].median())
    print(f"  {inst:<14} closing {a:+.4f} | draws_readoff {b:+.4f} | match={ok} | "
          f"draw median {med:+.4f}")
    assert ok, f"WV2 mismatch {inst} — STOP"
    add("gate_wv2_match", inst, "WV2", a, notes=f"draws_readoff {b:+.4f}; draw median {med:+.4f}")
print("GATE PASSED")

def pt(inst, w):
    return float(eq.loc[(inst, w), "estimate"])

def stats_from(d, point):
    d = d[~np.isnan(d)]
    lo = (1 + (d <= 0).sum()) / (len(d) + 1)
    hi = (1 + (d >= 0).sum()) / (len(d) + 1)
    return dict(est=point, se=float(d.std(ddof=1)),
                ci90=(float(np.quantile(d, .05)), float(np.quantile(d, .95))),
                ci95=(float(np.quantile(d, .025)), float(np.quantile(d, .975))),
                p=min(1.0, 2 * min(lo, hi)), n=int(len(d)))


parquet shape: (34958, 5) (expected (34958, 5))

WV2 LEVELs across tables (must match to 4 dp):
  military       closing -0.7631 | draws_readoff -0.7631 | match=True | draw median -0.7648
  police         closing -0.7604 | draws_readoff -0.7604 | match=True | draw median -0.7652
  courts         closing -0.5270 | draws_readoff -0.5270 | match=True | draw median -0.5313
  parliament     closing -0.9259 | draws_readoff -0.9259 | match=True | draw median -0.9437
  government     closing -0.4292 | draws_readoff -0.4292 | match=True | draw median -0.4314
  civil_service  closing -0.7832 | draws_readoff -0.7832 | match=True | draw median -0.7914
GATE PASSED


## 1 — Is WV6 an excursion or a wobble?

Replicate-paired neighbour contrasts for the army; police as the other partial-fade comparison
case; courts as negative control. Joint test: χ²(2) on the 2-vector [WV6−WV5, WV6−WV7] with its
bootstrap covariance.

In [2]:
# 1a-1d
for inst in ["military", "police", "courts"]:
    d65 = (piv[(inst, "WV6")] - piv[(inst, "WV5")]).dropna()
    d67 = (piv[(inst, "WV6")] - piv[(inst, "WV7")]).dropna()
    s65 = stats_from(d65.values, pt(inst, "WV6") - pt(inst, "WV5"))
    s67 = stats_from(d67.values, pt(inst, "WV6") - pt(inst, "WV7"))
    dm = (piv[(inst, "WV6")] - (piv[(inst, "WV5")] + piv[(inst, "WV7")]) / 2).dropna()
    sm = stats_from(dm.values, pt(inst, "WV6") - (pt(inst, "WV5") + pt(inst, "WV7")) / 2)
    D = pd.concat([d65, d67], axis=1).dropna().values
    dvec = np.array([s65["est"], s67["est"]])
    V = np.cov(D.T)
    c2 = float(dvec @ np.linalg.inv(V) @ dvec)
    pj = float(1 - stats.chi2.cdf(c2, 2))
    role = {"military": "", "police": " (comparison case)",
            "courts": " (negative control)"}[inst]
    print(f"{inst}{role}:")
    for lbl, s in [("WV6-WV5", s65), ("WV6-WV7", s67), ("WV6-mean(WV5,WV7)", sm)]:
        print(f"  {lbl:<18} {s['est']:+.4f} (SE {s['se']:.4f}) "
              f"90%[{s['ci90'][0]:+.3f},{s['ci90'][1]:+.3f}] "
              f"95%[{s['ci95'][0]:+.3f},{s['ci95'][1]:+.3f}] p={s['p']:.4f} n={s['n']}")
        add(f"wv6_excursion_{lbl}", inst, "WV6", s["est"], s["se"], s["ci90"],
            s["ci95"], s["p"], s["n"], notes="replicate-paired")
    print(f"  joint chi2(2) = {c2:.2f}  p = {pj:.4f}\n")
    add("wv6_excursion_joint_chi2_2df", inst, "WV6", c2, p=pj,
        n=len(D), notes="H0: WV6 level equals both neighbours; bootstrap covariance")


military:
  WV6-WV5            -0.2435 (SE 0.2302) 90%[-0.602,+0.139] 95%[-0.668,+0.210] p=0.3217 n=1000
  WV6-WV7            -0.2513 (SE 0.1640) 90%[-0.507,+0.033] 95%[-0.543,+0.084] p=0.1459 n=1000
  WV6-mean(WV5,WV7)  -0.2474 (SE 0.1727) 90%[-0.512,+0.043] 95%[-0.550,+0.095] p=0.1858 n=1000
  joint chi2(2) = 2.44  p = 0.2952

police (comparison case):
  WV6-WV5            -0.0867 (SE 0.1868) 90%[-0.390,+0.233] 95%[-0.430,+0.289] p=0.6273 n=997
  WV6-WV7            -0.3978 (SE 0.1556) 90%[-0.660,-0.141] 95%[-0.729,-0.106] p=0.0120 n=997
  WV6-mean(WV5,WV7)  -0.2422 (SE 0.1521) 90%[-0.495,+0.010] 95%[-0.543,+0.053] p=0.1162 n=997
  joint chi2(2) = 8.04  p = 0.0179

courts (negative control):
  WV6-WV5            -0.0119 (SE 0.2031) 90%[-0.333,+0.322] 95%[-0.396,+0.400] p=0.9990 n=1000
  WV6-WV7            -0.0134 (SE 0.1674) 90%[-0.295,+0.260] 95%[-0.338,+0.303] p=0.9810 n=1000
  WV6-mean(WV5,WV7)  -0.0126 (SE 0.1662) 90%[-0.285,+0.255] 95%[-0.324,+0.315] p=0.9750 n=1000
  joint chi2(

## 2 — The "bloc" premise at 1990

Six WV2 LEVELs with p vs zero; homogeneity χ²(5) from the bootstrap covariance of the paired
6-vector (contrasts vs the army; complete replicates only); observed spread (max−min) and
between-institution SD (sample SD, ddof = 1, computed within each replicate). Same battery at
WV7. Note: replicate-level spread measures include sampling-noise inflation, so their CIs sit
high relative to the full-sample point — read them as conservative upper bands.

In [3]:
# 2a-2d
for w in ["WV2", "WV7"]:
    lv = piv.loc[:, [(i, w) for i in INST6]].dropna()
    pts = np.array([pt(i, w) for i in INST6])
    tab = pd.DataFrame({
        "institution": INST6, "LEVEL": np.round(pts, 4),
        "boot_se": [round(float(eq.loc[(i, w), "se"]), 4) for i in INST6],
        "ci95": [f"[{eq.loc[(i, w), 'ci95_lo']:+.3f},{eq.loc[(i, w), 'ci95_hi']:+.3f}]"
                 for i in INST6],
        "p_vs_zero": [round(float(eq.loc[(i, w), "p"]), 4) for i in INST6]})
    nz = int((tab["p_vs_zero"] < .05).sum())
    print(f"--- {w}:")
    print(tab.to_string(index=False))
    print(f"institutions excluding zero (p<.05): {nz}/6")
    C = np.zeros((5, 6))
    for k in range(5):
        C[k, 0] = 1
        C[k, k + 1] = -1
    dvec = C @ pts
    V = C @ np.cov(lv.values.T) @ C.T
    c2 = float(dvec @ np.linalg.inv(V) @ dvec)
    pj = float(1 - stats.chi2.cdf(c2, 5))
    spread_pt = float(pts.max() - pts.min())
    sd_pt = float(pts.std(ddof=1))
    spread_d = lv.values.max(axis=1) - lv.values.min(axis=1)
    sd_d = lv.values.std(axis=1, ddof=1)
    print(f"homogeneity chi2(5) = {c2:.2f}  p = {pj:.5f}  (complete replicates n={len(lv)})")
    print(f"spread max-min: {spread_pt:.4f}  95% [{np.quantile(spread_d, .025):.4f}, "
          f"{np.quantile(spread_d, .975):.4f}]")
    print(f"between-institution SD: {sd_pt:.4f}  95% [{np.quantile(sd_d, .025):.4f}, "
          f"{np.quantile(sd_d, .975):.4f}]\n")
    for i in INST6:
        add(f"level_{w}", i, w, pt(i, w), float(eq.loc[(i, w), "se"]),
            (float(eq.loc[(i, w), "ci90_lo"]), float(eq.loc[(i, w), "ci90_hi"])),
            (float(eq.loc[(i, w), "ci95_lo"]), float(eq.loc[(i, w), "ci95_hi"])),
            float(eq.loc[(i, w), "p"]), notes="copied from closing_checks_table")
    add(f"homogeneity_chi2_5df", "all_six", w, c2, p=pj, n=len(lv),
        notes="H0: all six LEVELs equal; bootstrap covariance, contrasts vs army")
    add(f"spread_max_minus_min", "all_six", w, spread_pt,
        ci95=(float(np.quantile(spread_d, .025)), float(np.quantile(spread_d, .975))),
        n=len(lv), notes="replicate-level spread includes sampling-noise inflation")
    add(f"between_inst_sd", "all_six", w, sd_pt,
        ci95=(float(np.quantile(sd_d, .025)), float(np.quantile(sd_d, .975))),
        n=len(lv), notes="sample SD ddof=1; replicate-level SD includes noise inflation")


--- WV2:
  institution   LEVEL  boot_se            ci95  p_vs_zero
     military -0.7631   0.1410 [-1.031,-0.483]      0.002
       police -0.7604   0.1481 [-1.065,-0.474]      0.002
       courts -0.5270   0.1607 [-0.843,-0.216]      0.004
   parliament -0.9259   0.1621 [-1.260,-0.609]      0.002
   government -0.4292   0.1534 [-0.717,-0.144]      0.004
civil_service -0.7832   0.1514 [-1.074,-0.496]      0.002
institutions excluding zero (p<.05): 6/6
homogeneity chi2(5) = 11.43  p = 0.04355  (complete replicates n=993)
spread max-min: 0.4967  95% [0.3189, 0.8700]
between-institution SD: 0.1838  95% [0.1203, 0.3188]

--- WV7:
  institution   LEVEL  boot_se            ci95  p_vs_zero
     military -0.0807   0.0938 [-0.252,+0.113]     0.3796
       police -0.1376   0.0894 [-0.308,+0.042]     0.1403
       courts -0.4367   0.0779 [-0.582,-0.282]     0.0020
   parliament -0.0764   0.0666 [-0.202,+0.053]     0.2320
   government -0.5550   0.0713 [-0.685,-0.406]     0.0020
civil_service -0.4

## 3 — Dispersion trajectory

**This dispersion series is a descriptive, post-hoc quantity computed after the confirmatory
analyses** — labelled as such in the table's notes. Two series: (i) all institutions available
in the wave (courts absent from WV4 — noted); (ii) the five non-courts institutions, comparable
across all six waves. Paired SD differences use the five-institution series so WV4 is included.

In [4]:
# 3a-3c
sd5_draws, DISP = {}, {}
for w in WAVES:
    avail = [i for i in INST6 if (i, w) in eq.index]
    for label, insts in [("all_available", avail), ("five_noncourts", INST5)]:
        lv = piv.loc[:, [(i, w) for i in insts]].dropna()
        pts = np.array([pt(i, w) for i in insts])
        sd_pt = float(pts.std(ddof=1))
        sd_d = lv.values.std(axis=1, ddof=1)
        ci95 = (float(np.quantile(sd_d, .025)), float(np.quantile(sd_d, .975)))
        if label == "five_noncourts":
            sd5_draws[w] = pd.Series(sd_d, index=lv.index)
            DISP[w] = (sd_pt, ci95)
        note = ("DESCRIPTIVE POST-HOC; " +
                ("courts absent WV4; " if w == "WV4" and label == "all_available" else "") +
                f"{len(insts)} institutions; replicate SDs include noise inflation")
        print(f"{w} {label:<15} ({len(insts)} inst): SD {sd_pt:.4f} "
              f"95% [{ci95[0]:.4f}, {ci95[1]:.4f}]  n={len(lv)}")
        add(f"dispersion_sd_{label}", "cross_institution", w, sd_pt,
            ci95=ci95, n=len(lv), notes=note)

pt5 = {w: float(np.array([pt(i, w) for i in INST5]).std(ddof=1)) for w in WAVES}
print("\n3b — paired SD differences (five-institution series):")
for a, b in [("WV7", "WV2"), ("WV4", "WV2"), ("WV7", "WV4")]:
    dd = (sd5_draws[a] - sd5_draws[b]).dropna().values
    s = stats_from(dd, pt5[a] - pt5[b])
    print(f"  SD({a})-SD({b}): {s['est']:+.4f} (SE {s['se']:.4f}) "
          f"90%[{s['ci90'][0]:+.3f},{s['ci90'][1]:+.3f}] "
          f"95%[{s['ci95'][0]:+.3f},{s['ci95'][1]:+.3f}] p={s['p']:.4f} n={s['n']}")
    add(f"dispersion_sd_diff_{a}-{b}", "cross_institution", f"{a}-{b}", s["est"],
        s["se"], s["ci90"], s["ci95"], s["p"], s["n"],
        notes="DESCRIPTIVE POST-HOC; five non-courts institutions, paired")
mn = min(pt5, key=pt5.get)
print(f"\npoint SD series (five-inst): {({w: round(pt5[w], 4) for w in WAVES})}")
print(f"3c — dispersion in 2018 vs 1990: SD(WV7)-SD(WV2) = {pt5['WV7']-pt5['WV2']:+.4f}, "
      f"p = 0.831 -> NOT significantly larger. Series minimum: {mn} (1996), not WV4 (2001); "
      f"WV4 is second-lowest (0.0869 vs {pt5[mn]:.4f}).")


WV2 all_available   (6 inst): SD 0.1838 95% [0.1203, 0.3188]  n=993
WV2 five_noncourts  (5 inst): SD 0.1828 95% [0.1131, 0.3345]  n=993
WV3 all_available   (6 inst): SD 0.0767 95% [0.0562, 0.2024]  n=993
WV3 five_noncourts  (5 inst): SD 0.0751 95% [0.0496, 0.2145]  n=993
WV4 all_available   (5 inst): SD 0.0869 95% [0.0491, 0.1944]  n=993
WV4 five_noncourts  (5 inst): SD 0.0869 95% [0.0491, 0.1944]  n=993
WV5 all_available   (6 inst): SD 0.1775 95% [0.1030, 0.3267]  n=993
WV5 five_noncourts  (5 inst): SD 0.1968 95% [0.1039, 0.3556]  n=993
WV6 all_available   (6 inst): SD 0.0803 95% [0.0540, 0.2046]  n=993
WV6 five_noncourts  (5 inst): SD 0.0897 95% [0.0504, 0.2224]  n=993
WV7 all_available   (6 inst): SD 0.2120 95% [0.1723, 0.2718]  n=993
WV7 five_noncourts  (5 inst): SD 0.2223 95% [0.1772, 0.2853]  n=993

3b — paired SD differences (five-institution series):
  SD(WV7)-SD(WV2): +0.0395 (SE 0.0623) 90%[-0.092,+0.115] 95%[-0.117,+0.131] p=0.8310 n=993
  SD(WV4)-SD(WV2): -0.0959 (SE 0.0675

## Final table — main result table with dispersion rows appended

Six institutions × six waves (LEVEL, 95% CI, both equivalence decisions), unchanged from
`draws_readoff.ipynb`; dispersion rows (descriptive, post-hoc) appended per wave.

In [5]:
# reprint main table + dispersion rows; write CSV
DELTA_BASE = abs(pt("military", "WV2"))
final = []
for inst in INST6:
    row = {"institution": inst}
    for w in WAVES:
        if (inst, w) not in eq.index:
            row[w] = "— (not fielded)"
            continue
        r_ = eq.loc[(inst, w)]
        dec_b = ("eq" if (r_["ci90_lo"] > -DELTA_BASE and r_["ci90_hi"] < DELTA_BASE)
                 else "NON-eq")
        dec_c = "eq" if r_["decision_courts"] == "EQUIVALENT" else "NON-eq"
        row[w] = (f"{r_['estimate']:+.3f} [{r_['ci95_lo']:+.3f},{r_['ci95_hi']:+.3f}] "
                  f"@0.42:{dec_c} @0.76:{dec_b}")
    final.append(row)
disp_row = {"institution": "DISPERSION (SD, 5 non-courts; descriptive)"}
for w in WAVES:
    sd_pt, ci95 = DISP[w]
    disp_row[w] = f"{sd_pt:.3f} [{ci95[0]:.3f},{ci95[1]:.3f}]"
final.append(disp_row)
FINAL = pd.DataFrame(final).set_index("institution")
master = pd.DataFrame(ROWS, columns=["quantity", "institution", "wave", "estimate",
                                     "boot_se", "ci90_lo", "ci90_hi", "ci95_lo",
                                     "ci95_hi", "p", "n_draws", "notes"])
master.to_csv(f"{R}/draws_readoff2_table.csv", index=False)
print(f"wrote results/draws_readoff2_table.csv ({len(master)} rows)\n")
with pd.option_context("display.max_colwidth", 60, "display.width", 320):
    print(FINAL.to_string())


wrote results/draws_readoff2_table.csv (51 rows)

                                                                                         WV2                                           WV3                                       WV4                                               WV5                                           WV6                                           WV7
institution                                                                                                                                                                                                                                                                                                                       
military                                    -0.763 [-1.031,-0.483] @0.42:NON-eq @0.76:NON-eq  -0.336 [-0.577,-0.100] @0.42:NON-eq @0.76:eq  -0.025 [-0.221,+0.177] @0.42:eq @0.76:eq          -0.089 [-0.451,+0.285] @0.42:eq @0.76:eq  -0.332 [-0.590,-0.048] @0.42:NON-eq @0.76:eq      -0.081 [-0.252,+0.113]